In [0]:
# Cell 1 — Setup
from databricks.sdk import WorkspaceClient
w        = WorkspaceClient()
username = w.current_user.me().user_name.split("@")[0].replace(".", "_").replace("-", "_")
catalog  = "bootcamp_students"
schema   = f"maintops"
print(f"Working in: {catalog}.{schema}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.raw_faq (
  doc_id            STRING    COMMENT 'Unique document identifier (UUID)',
  source_type       STRING    COMMENT 'pdf | web | image',
  source_url        STRING    COMMENT 'File path or URL the content was extracted from',
  title             STRING    COMMENT 'Human-readable document title',
  content           STRING    COMMENT 'Extracted plain text content',
  ingested_at       TIMESTAMP COMMENT 'Ingestion timestamp'
)
TBLPROPERTIES (delta.enableChangeDataFeed = true)
"""
)

In [0]:
PDF_FILENAME   = "MaintOps_Public_User_Guide.pdf"
VOL_PATH       = f"/Volumes/{catalog}/{schema}/maintops_docs/"
PDF_VOL_PATH   = f"{VOL_PATH}{PDF_FILENAME}"

In [0]:
spark.sql(f"""
INSERT INTO {catalog}.{schema}.raw_faq

WITH parsed_documents AS (
  SELECT
    path,
    ai_parse_document(
      content,
      map(
        'imageOutputPath', '/Volumes/{catalog}/{schema}/maintops_docs/',
        'descriptionElementTypes', '*'
      )
    ) AS parsed
  FROM READ_FILES('{PDF_VOL_PATH}', format => 'binaryFile')
),
parsed_text AS (
  SELECT
    path,
    parsed:document as json_content,
    concat_ws(
      '\n\n',
      transform(
        try_cast(parsed:document:elements AS ARRAY<VARIANT>),
        element -> try_cast(element:content AS STRING)
      )
    ) AS text
  FROM parsed_documents
  WHERE try_cast(parsed:error_status AS STRING) IS NULL
)
SELECT
  uuid()                                                  AS doc_id,
  'pdf'                                                   AS source_type,
  '{PDF_VOL_PATH}'                                        AS source_url,
  'Maintops FAQ'                                          AS title,
  parsed_text.text                                        AS content,
  current_timestamp()                                     AS ingested_at
  
FROM
  parsed_text
""")

display(spark.table(f"{catalog}.{schema}.raw_faq"))